In [2]:
from typing import TypedDict

from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, StateGraph


In [3]:
load_dotenv()

llm = ChatGroq(model='openai/gpt-oss-20b')

In [4]:
class JokeState(TypedDict):

    topic: str
    joke: str
    explanation: str

In [5]:
def generate_joke(state: JokeState):

    prompt = f'generate a joke on the topic {state["topic"]}'
    response = llm.invoke(prompt).content

    return {'joke': response}

In [6]:
def generate_explanation(state: JokeState):

    prompt = f'write an explanation for the joke - {state["joke"]}'
    response = llm.invoke(prompt).content

    return {'explanation': response}

In [7]:
graph = StateGraph(JokeState)

graph.add_node('generate_joke', generate_joke)
graph.add_node('generate_explanation', generate_explanation)

graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'generate_explanation')
graph.add_edge('generate_explanation', END)

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)

In [20]:
config1 = {"configurable": {"thread_id": "1"}}
workflow.invoke({'topic':'pizza'}, config=config1)

{'topic': 'pizza',
 'joke': 'Why did the pizza apply for a job?\n\nBecause it wanted to earn a *crust* of a living! 🍕😄',
 'explanation': '**Explanation of the joke**\n\n> “Why did the pizza apply for a job?  \n>  Because it wanted to earn a *crust* of a living! 🍕😄”\n\nThe joke is a classic **pun** that mixes a literal pizza term with a common idiom.\n\n| Element | What it is | How it works in the joke |\n|---------|------------|--------------------------|\n| **Pizza “crust”** | The outer, baked edge of a pizza. | It’s a concrete, pizza‑specific word that the joke uses to keep the humor grounded in the pizza world. |\n| **“Earn a living”** | The idiom meaning to make money to support oneself. | The phrase is the target of the pun. |\n| **“Crust of a living”** | A playful twist that replaces “a living” with the pizza’s crust. | The word “crust” sounds like the word “crust” (the same spelling) but is inserted into the idiom, making it a *homophonic* play on “earn a living.” |\n\n### How t

In [21]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza apply for a job?\n\nBecause it wanted to earn a *crust* of a living! 🍕😄', 'explanation': '**Explanation of the joke**\n\n> “Why did the pizza apply for a job?  \n>  Because it wanted to earn a *crust* of a living! 🍕😄”\n\nThe joke is a classic **pun** that mixes a literal pizza term with a common idiom.\n\n| Element | What it is | How it works in the joke |\n|---------|------------|--------------------------|\n| **Pizza “crust”** | The outer, baked edge of a pizza. | It’s a concrete, pizza‑specific word that the joke uses to keep the humor grounded in the pizza world. |\n| **“Earn a living”** | The idiom meaning to make money to support oneself. | The phrase is the target of the pun. |\n| **“Crust of a living”** | A playful twist that replaces “a living” with the pizza’s crust. | The word “crust” sounds like the word “crust” (the same spelling) but is inserted into the idiom, making it a *homophonic* play on “earn a livi

In [22]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza apply for a job?\n\nBecause it wanted to earn a *crust* of a living! 🍕😄', 'explanation': '**Explanation of the joke**\n\n> “Why did the pizza apply for a job?  \n>  Because it wanted to earn a *crust* of a living! 🍕😄”\n\nThe joke is a classic **pun** that mixes a literal pizza term with a common idiom.\n\n| Element | What it is | How it works in the joke |\n|---------|------------|--------------------------|\n| **Pizza “crust”** | The outer, baked edge of a pizza. | It’s a concrete, pizza‑specific word that the joke uses to keep the humor grounded in the pizza world. |\n| **“Earn a living”** | The idiom meaning to make money to support oneself. | The phrase is the target of the pun. |\n| **“Crust of a living”** | A playful twist that replaces “a living” with the pizza’s crust. | The word “crust” sounds like the word “crust” (the same spelling) but is inserted into the idiom, making it a *homophonic* play on “earn a liv

In [15]:
config2 = {"configurable": {"thread_id": "2"}}
workflow.invoke({'topic':'pasta'}, config=config2)

{'topic': 'pasta',
 'joke': 'Why did the spaghetti start a dating app?\n\nBecause it heard that good relationships are all about **pasta** and **connection**—but it kept getting swiped left by all the *spaghetti*‑no‑one!',
 'explanation': '**Why the joke works – a word‑play breakdown**\n\n| Part of the joke | What it means | Why it’s funny |\n|------------------|----------------|----------------|\n| **“Why did the spaghetti start a dating app?”** | A silly, absurd premise: a piece of pasta is a person (or an app‑user) who wants to find a partner. | The absurdity of a non‑living food item behaving like a human gives the joke its initial hook. |\n| **“Because it heard that good relationships are all about **pasta** and **connection**…”** | 1. **Pasta** sounds like **past**, hinting at “past relationships” or “history” in a romance. 2. **Connection** is a double‑meaning: the internet connection needed for a dating app and the emotional bond in a relationship. | The joke turns a common phr

In [16]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza apply for a job?  \n\nBecause it wanted to *cheese* its way into a higher *slice* of life! 🍕😄', 'explanation': '**Why the joke is funny – a word‑play breakdown**\n\n| Part of the joke | Literal meaning | Pun / double meaning | How it creates the laugh |\n|-------------------|-----------------|----------------------|--------------------------|\n| **“Why did the pizza apply for a job?”** | A pizza, a food, is being asked to do a human activity (apply for a job). | It anthropomorphises the pizza, turning it into a character that can think and act. | The absurdity of a pizza having a career ambition sets the stage for the pun. |\n| **“Because it wanted to *cheese* its way”** | *Cheese* is the topping on a pizza. | “Cheese” is also a slang verb meaning *to make money* (e.g., “cheese that paycheck”). | The phrase “cheese its way” sounds like “cheese it’s way,” a playful way of saying “make money and get ahead.” |\n| **“into a

In [109]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza go to the doctor? Because it was feeling a little saucy!', 'explanation': 'This joke plays on the double meaning of the word "saucy." In one sense, "saucy" can mean bold, impertinent, or sassy. But in the context of a pizza, "saucy" refers to the tomato sauce typically used as a base on pizzas. So when the pizza went to the doctor because it was feeling "saucy," it implies that the pizza was not feeling well due to too much sauce, rather than being bold or sassy. The humor comes from the unexpected twist on the word\'s meaning.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f06cc6e-93a2-6a08-8002-395e36be0f5e'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}, 'thread_id': '1'}, created_at='2025-07-29T21:56:42.071296+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f06cc6e-7a2f-60ea-8001-4ac26c539f8d'}}, tasks=(), int

### Time Travel

In [18]:
workflow.get_state({"configurable": {"thread_id": "1", "checkpoint_id": "1f06cc6e-7232-6cb1-8000-f71609e6cec5"}})

StateSnapshot(values={}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_id': '1f06cc6e-7232-6cb1-8000-f71609e6cec5'}}, metadata=None, created_at=None, parent_config=None, tasks=(), interrupts=())

In [23]:
workflow.invoke(None, {"configurable": {"thread_id": "1", "checkpoint_id": "1f1d2ac4-d161-6aec-8000-b2be55d2a46c"}})

{'topic': 'pizza',
 'joke': 'Why did the pizza bring a ladder to the party?  \n\nBecause it heard the “cheese” was on a whole other level! 🍕😄',
 'explanation': '**Explanation of the joke**\n\n> **Why did the pizza bring a ladder to the party?  \n>  Because it heard the “cheese” was on a whole other level!** 🍕😄  \n\nThe punch‑line hinges on a double meaning of the word **“level.”**  \n\n1. **Figurative “level”** – In everyday speech, saying something is “on a whole other level” means it’s *superb,* *impressive,* or *better than usual.*  \n2. **Literal “level”** – On a pizza, the cheese sits on top of the dough, literally on a higher tier or “level” of the pie.\n\nSo when the pizza says it heard the cheese was “on a whole other level,” it’s playing with both senses:\n\n- **Metaphorically**: The cheese is great—“the best topping ever.”\n- **Literally**: The cheese is physically higher up on the pizza.\n\nBecause the pizza is anthropomorphized (treated as if it can think and act like a per

In [24]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza bring a ladder to the party?  \n\nBecause it heard the “cheese” was on a whole other level! 🍕😄', 'explanation': '**Explanation of the joke**\n\n> **Why did the pizza bring a ladder to the party?  \n>  Because it heard the “cheese” was on a whole other level!** 🍕😄  \n\nThe punch‑line hinges on a double meaning of the word **“level.”**  \n\n1. **Figurative “level”** – In everyday speech, saying something is “on a whole other level” means it’s *superb,* *impressive,* or *better than usual.*  \n2. **Literal “level”** – On a pizza, the cheese sits on top of the dough, literally on a higher tier or “level” of the pie.\n\nSo when the pizza says it heard the cheese was “on a whole other level,” it’s playing with both senses:\n\n- **Metaphorically**: The cheese is great—“the best topping ever.”\n- **Literally**: The cheese is physically higher up on the pizza.\n\nBecause the pizza is anthropomorphized (treated as if it can thin

#### Updating State

In [25]:
workflow.update_state({"configurable": {"thread_id": "1", "checkpoint_id": "1f1d2ac4-d161-6aec-8000-b2be55d2a46c", "checkpoint_ns": ""}}, {'topic':'samosa'})

{'configurable': {'thread_id': '1',
  'checkpoint_ns': '',
  'checkpoint_id': '1f1d2ad6-f979-69b9-8001-6abc45e9ce12'}}

In [26]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'samosa'}, next=('generate_joke',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1d2ad6-f979-69b9-8001-6abc45e9ce12'}}, metadata={'source': 'update', 'step': 1, 'parents': {}}, created_at='2026-10-28T08:56:16.857707+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1d2ac4-d161-6aec-8000-b2be55d2a46c'}}, tasks=(PregelTask(id='adc5f6d1-c4a2-7c21-2846-ddfa7ebcb8ad', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error=None, interrupts=(), state=None, result=None),), interrupts=()),
 StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza bring a ladder to the party?  \n\nBecause it heard the “cheese” was on a whole other level! 🍕😄', 'explanation': '**Explanation of the joke**\n\n> **Why did the pizza bring a ladder to the party?  \n>  Because it heard the “cheese” was on a whole other level!** 🍕😄  \n\nThe punch‑line hinges on a double meaning o

In [29]:
workflow.invoke(None, {"configurable": {"thread_id": "1", "checkpoint_id": "1f1d2ad6-f979-69b9-8001-6abc45e9ce12"}})

{'topic': 'samosa',
 'joke': 'Why did the samosa bring a ladder to the party?\n\nBecause it heard the snacks were *well* stacked and wanted to *rise* to the occasion!',
 'explanation': '**Explanation of the joke**\n\n> **Why did the samosa bring a ladder to the party?  \n>  Because it heard the snacks were *well* stacked and wanted to *rise* to the occasion!**\n\nThe humor comes from a few layers of wordplay and a little absurdity:\n\n| Element | What it’s doing | Why it’s funny |\n|---------|-----------------|----------------|\n| **Samosa** | A small, fried snack that normally doesn’t have any agency. | Imagining a samosa acting like a person (and bringing a ladder) is already silly. |\n| **“Well” stacked** | The phrase *well‑stacked* means “carefully arranged” or “in a good order.”  It can also be split into the noun *well* (a hole or pit) and the adjective *well* (meaning “good”). | The joke hints at both meanings: the snacks are stacked so neatly they could be a “well” (like a pit 

In [30]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'samosa', 'joke': 'Why did the samosa bring a ladder to the party?\n\nBecause it heard the snacks were *well* stacked and wanted to *rise* to the occasion!', 'explanation': '**Explanation of the joke**\n\n> **Why did the samosa bring a ladder to the party?  \n>  Because it heard the snacks were *well* stacked and wanted to *rise* to the occasion!**\n\nThe humor comes from a few layers of wordplay and a little absurdity:\n\n| Element | What it’s doing | Why it’s funny |\n|---------|-----------------|----------------|\n| **Samosa** | A small, fried snack that normally doesn’t have any agency. | Imagining a samosa acting like a person (and bringing a ladder) is already silly. |\n| **“Well” stacked** | The phrase *well‑stacked* means “carefully arranged” or “in a good order.”  It can also be split into the noun *well* (a hole or pit) and the adjective *well* (meaning “good”). | The joke hints at both meanings: the snacks are stacked so neatly they could be a

### Fault Tolerance

In [2]:
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import InMemorySaver
from typing import TypedDict
import time

In [3]:
# 1. Define the state
class CrashState(TypedDict):
    input: str
    step1: str
    step2: str

In [4]:
# 2. Define steps
def step_1(state: CrashState) -> CrashState:
    print("✅ Step 1 executed")
    return {"step1": "done", "input": state["input"]}

def step_2(state: CrashState) -> CrashState:
    print("⏳ Step 2 hanging... now manually interrupt from the notebook toolbar (STOP button)")
    time.sleep(1000)  # Simulate long-running hang
    return {"step2": "done"}

def step_3(state: CrashState) -> CrashState:
    print("✅ Step 3 executed")
    return {"done": True}

In [5]:
# 3. Build the graph
builder = StateGraph(CrashState)
builder.add_node("step_1", step_1)
builder.add_node("step_2", step_2)
builder.add_node("step_3", step_3)

builder.set_entry_point("step_1")
builder.add_edge("step_1", "step_2")
builder.add_edge("step_2", "step_3")
builder.add_edge("step_3", END)

checkpointer = InMemorySaver()
graph = builder.compile(checkpointer=checkpointer)

In [ ]:
try:
    print("▶️ Running graph: Please manually interrupt during Step 2...")
    graph.invoke({"input": "start"}, config={"configurable": {"thread_id": 'thread-1'}})
except KeyboardInterrupt:
    print("❌ Kernel manually interrupted (crash simulated).")

▶️ Running graph: Please manually interrupt during Step 2...
✅ Step 1 executed
⏳ Step 2 hanging... now manually interrupt from the notebook toolbar (STOP button)


In [ ]:
# 6. Re-run to show fault-tolerant resume
print("\n🔁 Re-running the graph to demonstrate fault tolerance...")
final_state = graph.invoke(None, config={"configurable": {"thread_id": 'thread-1'}})
print("\n✅ Final State:", final_state)

In [ ]:
list(graph.get_state_history({"configurable": {"thread_id": 'thread-1'}}))